#### Imports

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import KMRF class
from kmrf import KMRF
from KMRF_training_config import *
from CODE_data_collection_and_processing.ASSET_SYMBOLS import *

# Import parallelization tools
from joblib import Parallel, delayed
import multiprocessing

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Get number of CPUs
n_cpus = multiprocessing.cpu_count()
n_jobs = max(1, n_cpus // 2)  # Use half of available CPUs

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")
print(f"  Available CPUs: {n_cpus}")
print(f"  Using {n_jobs} CPUs for parallel training")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4
  Available CPUs: 8
  Using 4 CPUs for parallel training


## Getting Price Data

#### -------------------------------------------------------------------------------------------------

In [3]:
# Prepare data
international_index_symbol_names = pd.read_csv('data/inputs/fmp_index_list.csv').set_index('symbol')['name']
international_index_symbol_names = international_index_symbol_names[~international_index_symbol_names.index.isin(['^GSPC', '^NDX'])].to_dict()
commodity_symbol_names = pd.read_csv('data/inputs/fmp_commodity_list.csv').set_index('symbol')['name'].to_dict()

'''
etf_symbol_names = {
    # BOND ETFS
    # 'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    # 'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    # 'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    
    # Commodity ETFs
    'GLD': 'SPDR Gold Shares',
    'SLV': 'iShares Silver Trust',
    'USO': 'United States Oil Fund',
    'UNG': 'United States Natural Gas Fund',
    'DBA': 'Invesco DB Agriculture',
    'CPER': 'United States Copper Index Fund',
    'DBB': 'Invesco DB Base Metals',
    'PALL': 'abrdn Palladium Shares ETF',
    'PLTM': 'GraniteShares Platinum Trust',
    'WEAT': 'Teucrium Wheat Fund',
    'SOYB': 'Teucrium Soybean Fund',
    'CORN': 'Teucrium Corn Fund',
    'CANE': 'Teucrium Sugar Fund',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    # 'VOO': 'Vanguard S&P 500 ETF',
    # 'RSP': 'Invesco S&P 500 Equal Weight ETF',
    # 'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'IWM': 'iShares Russell 2000 ETF',
    # 'IWB': 'iShares Russell 1000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    # 'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    # 'XLRE': 'Real Estate Select Sector SPDR',
    # 'XLC': 'Communication Services Select Sector SPDR',
    'IYR': 'iShares U.S. Real Estate ETF',
    'IYZ': 'iShares U.S. Telecommunications ETF',
    
    # GROWTH ETFs
    'MGK': 'Vanguard Mega Cap Growth ETF',
    'IVW': 'iShares S&P 500 Growth ETF',
    # 'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    # 'VUG': 'Vanguard Growth ETF',
    
    # VALUE ETFs
    'MGV': 'Vanguard Mega Cap Value ETF',
    'IVE': 'iShares S&P 500 Value ETF',
    # 'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    # 'VTV': 'Vanguard Value ETF',
    
    # SIZE ETFs
    'OEF': 'iShares S&P 100 ETF',
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    # 'IJH': 'iShares Core S&P Mid-Cap ETF',
    # 'IJR': 'iShares Core S&P Small-Cap ETF',
    # 'MDY': 'SPDR S&P MidCap 400 ETF',
    
    # INTERNATIONAL
    # 'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
    # 'EFA': 'iShares MSCI EAFE ETF',
    'EEM': 'iShares MSCI Emerging Markets ETF',
}

universe_symbol_names = {
    'IVV': 'IVV - iShares Core S&P 500 ETF',
    'IJH': 'IJH - iShares Core S&P Mid-Cap ETF',
    'IWM': 'IWM - iShares Russell 2000 ETF',
    'EFA': 'EFA - iShares MSCI EAFE ETF',
    'EEM': 'EEM - iShares MSCI Emerging Markets ETF',
    'AGG': 'AGG - iShares Core U.S. Aggregate Bond ETF',
    'SPTL': 'SPTL - SPDR Portfolio Long Term Treasury ETF',
    'HYG': 'HYG - iShares iBoxx $ High Yield Corporate Bond ETF',
    'SPBO': 'SPBO - SPDR Portfolio Corporate Bond ETF',
    'IYR': 'IYR - iShares U.S. Real Estate ETF',
    'DBC': 'DBC - Invesco DB Commodity Index Tracking Fund',
    'GLD': 'GLD - SPDR Gold Shares',
}
'''

# international_index_data = pd.read_csv('data/processed/index_data.csv', index_col=0, header=[0, 1], parse_dates=True)
# commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0, 1], parse_dates=True)
etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
# universe_data = pd.read_csv('data/processed/universe_etfs.csv', index_col=0, header=[0, 1], parse_dates=True)

# commodity_data_close_cols = commodity_data.columns[commodity_data.columns.get_level_values(1) == 'close']
# commodity_close_prices = commodity_data[commodity_data_close_cols].droplevel(1, axis=1).rename(columns=commodity_symbol_names)
# commodity_close_prices.columns = [col.replace('/', ' ') for col in commodity_close_prices.columns]

etf_close_cols = etf_data.columns[etf_data.columns.get_level_values(1) == 'close']
etf_close_prices = etf_data[etf_close_cols].droplevel(1, axis=1).rename(columns=ETF_SYMBOL_NAMES)

# universe_close_cols = universe_data.columns[universe_data.columns.get_level_values(1) == 'close']
# universe_close_prices = universe_data[universe_close_cols].droplevel(1, axis=1).rename(columns=universe_symbol_names)

In [4]:
etf_always_disclude = ['Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
etf_disclude = [name for name in etf_close_prices.columns if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']

etf_include = list(set(etf_close_prices.columns.tolist()) - set(etf_always_disclude) - set(etf_disclude))
etf_close_prices = etf_close_prices[etf_include]

# us_treasury = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']
# int_equity = ['Vanguard Total International Stock ETF', 'Vanguard FTSE Developed Markets ETF',\
#                                         'Vanguard FTSE Emerging Markets ETF','Vanguard FTSE Europe ETF',\
#                                         'Vanguard FTSE Pacific ETF', 'iShares China Large-Cap ETF',\
#                                         'iShares MSCI Japan ETF', 'iShares MSCI India ETF']
us_equity = list(set(etf_include))

#### -------------------------------------------------------------------------------------------------

In [7]:
# Create DataFrame of asset names
asset_names_df = pd.DataFrame({
    'universe': get_assets_by_class('universe') + ['']*7,
    'us_equity': get_assets_by_class('us_equity'),
    'commodity': get_assets_by_class('commodity') + ['']*5,
    'int_equity': get_assets_by_class('int_equity') + ['']*11,

})

from pandas import option_context
with option_context('display.max_colwidth', None):
    display(asset_names_df)

,universe,us_equity,commodity,int_equity
0,IVV - iShares Core S&P 500 ETF,SPDR S&P 500 ETF,Gold Futures,Vanguard Total International Stock ETF
1,IJH - iShares Core S&P Mid-Cap ETF,Invesco QQQ Trust,Wheat Futures,Vanguard FTSE Developed Markets ETF
2,IWM - iShares Russell 2000 ETF,iShares Russell 2000 ETF,Corn Futures,Vanguard FTSE Emerging Markets ETF
3,EFA - iShares MSCI EAFE ETF,SPDR Dow Jones Industrial Average ETF,Copper,Vanguard FTSE Europe ETF
4,EEM - iShares MSCI Emerging Markets ETF,Energy Select Sector SPDR,Sugar,Vanguard FTSE Pacific ETF
5,AGG - iShares Core U.S. Aggregate Bond ETF,Financial Select Sector SPDR,Silver Futures,iShares China Large-Cap ETF
6,SPTL - SPDR Portfolio Long Term Treasury ETF,Utilities Select Sector SPDR,US Dollar,iShares MSCI Japan ETF
7,HYG - iShares iBoxx $ High Yield Corporate Bond ETF,Industrial Select Sector SPDR,Soybean Futures,iShares MSCI India ETF
8,SPBO - SPDR Portfolio Corporate Bond ETF,Health Care Select Sector SPDR,Lumber Futures,
9,IYR - iShares U.S. Real Estate ETF,Technology Select Sector SPDR,Live Cattle Futures,


In [8]:
df = etf_close_prices['SPDR S&P 500 ETF'].to_frame().dropna()
rebal_dates = df.loc['2018-12-31':].index[::21]
rebal_dates

DatetimeIndex(['2018-12-31', '2019-01-31', '2019-03-04', '2019-04-02',
               '2019-05-02', '2019-06-03', '2019-07-02', '2019-08-01',
               '2019-08-30', '2019-10-01', '2019-10-30', '2019-11-29',
               '2019-12-31', '2020-01-31', '2020-03-03', '2020-04-01',
               '2020-05-01', '2020-06-02', '2020-07-01', '2020-07-31',
               '2020-08-31', '2020-09-30', '2020-10-29', '2020-11-30',
               '2020-12-30', '2021-02-01', '2021-03-03', '2021-04-01',
               '2021-05-03', '2021-06-02', '2021-07-01', '2021-08-02',
               '2021-08-31', '2021-09-30', '2021-10-29', '2021-11-30',
               '2021-12-30', '2022-01-31', '2022-03-02', '2022-03-31',
               '2022-05-02', '2022-06-01', '2022-07-01', '2022-08-02',
               '2022-08-31', '2022-09-30', '2022-10-31', '2022-11-30',
               '2022-12-30', '2023-02-01', '2023-03-03', '2023-04-03',
               '2023-05-03', '2023-06-02', '2023-07-05', '2023-08-03',
      

In [ ]:
from pathlib import Path

rebal_date = '20251007'
model_path = Path('saved_models/KAMA_MSR') / 'us_equity' / rebal_date

model_files = list(model_path.glob('*.pkl'))
model_files

# get asset names from model files
assets = [f.stem.split('_')[0] for f in model_files]
assets

['SPDR Gold Shares',
 'Consumer Discretionary Select Sector SPDR',
 'Consumer Staples Select Sector SPDR',
 'iShares Micro-Cap ETF',
 'iShares Russell 2000 ETF',
 'Financial Select Sector SPDR',
 'iShares S&P 500 Value ETF',
 'Invesco DB Agriculture',
 'Health Care Select Sector SPDR',
 'iShares U.S. Real Estate ETF',
 'iShares Russell 2000 Value ETF',
 'iShares S&P 500 Growth ETF',
 'Invesco QQQ Trust',
 'United States Natural Gas Fund',
 'SPDR Dow Jones Industrial Average ETF',
 'SPDR S&P 500 ETF',
 'iShares Russell 2000 Growth ETF',
 'iShares Russell Mid-Cap ETF',
 'Technology Select Sector SPDR',
 'United States Oil Fund',
 'Energy Select Sector SPDR',
 'Materials Select Sector SPDR',
 'Utilities Select Sector SPDR',
 'iShares Silver Trust',
 'Industrial Select Sector SPDR']

## Batch Train KMRF Model

In [13]:
def train_single_asset(asset_name, rebal_date):
    """Train KMRF model for a single asset."""
    try:
        print(f"Starting training for {asset_name}...")
        
        TRAINING_CONFIG = KMRF_Training_Config(
            asset_name='SPDR S&P 500 ETF',
            classification_type='original',
            use_data_type='master',
            end_date=rebal_date,
            feature_window_size=1,
            feature_asset_classes=[],
            cross_asset_specific=[],  # empty means all (only referring to universe assets)
            use_boruta_selection=False,
            use_consensus_selection=False
        )

        model = KMRF(
            asset_class='us_equity',
            asset_name=asset_name,
            classification_type='original',
            end_date=rebal_date,
            use_data_type='master',
            feature_window_size=1,  
            feature_asset_classes=[],
            cross_asset_specific=[],
            xgb_params=TRAINING_CONFIG.get_xgb_params(),
            use_boruta_selection=False,
            use_consensus_selection=False,
        )

        model.pipeline(optimize=False)

        path = f'saved_models/KMRF_new/{model.classification_type}/{model.asset_class}/{rebal_date}/'
        path += f'{model.asset_name}_KMRF_model.pkl'
        model.save_model(path)
        
        print(f"✓ Completed training for {asset_name}")
        return {'asset_name': asset_name, 'status': 'success', 'path': path}
        
    except Exception as e:
        print(f"✗ Error training {asset_name}: {str(e)}")
        return {'asset_name': asset_name, 'status': 'failed', 'error': str(e)}

rebal_date = '20251007'
n_jobs=1
# Parallel training
for asset_class in ['us_equity']:
    from pathlib import Path
    model_path = Path('saved_models/KAMA_MSR') / 'us_equity' / rebal_date
    model_files = list(model_path.glob('*.pkl'))
    assets = [f.stem.split('_')[0] for f in model_files]
    
    print(f"\n{'='*80}")
    print(f"TRAINING {len(assets)} ASSETS IN PARALLEL ({n_jobs} workers)")
    print(f"{'='*80}\n")
    
    # Run parallel training
    results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(train_single_asset)(asset_name, rebal_date) 
        for asset_name in assets
    )
    
    # Summary
    successful = sum(1 for r in results if r['status'] == 'success')
    failed = sum(1 for r in results if r['status'] == 'failed')
    
    print(f"\n{'='*80}")
    print(f"TRAINING COMPLETE")
    print(f"{'='*80}")
    print(f"✓ Successful: {successful}/{len(assets)}")
    print(f"✗ Failed: {failed}/{len(assets)}")
    
    if failed > 0:
        print("\nFailed assets:")
        for r in results:
            if r['status'] == 'failed':
                print(f"  - {r['asset_name']}: {r['error']}")


TRAINING 25 ASSETS IN PARALLEL (1 workers)

Starting training for SPDR Gold Shares...
KMRF model initialized
  Asset: SPDR Gold Shares
  Asset class: us_equity
  Classification type: original
  Training end date: 20251007
  Test end date: All available data
  Data type: master
  Data path: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
  KAMA+MSR model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20251007
  Validation/Test split: Will be calculated after loading data
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
  Feature selection: Boruta=False, Consensus=False
  Custom XGB parameters: {'n_estimators': 220, 'max_depth': 13, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.25, 'min_child_weight': 95, 'gamma': 0.045, 'random_state': 1010, 'n_jobs': -1, 'tree_method': 'hist', 'enable_categorical': False}

KMRF PIPELINE FOR SPDR Gold S

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.2s


Loaded data for: Consumer Discretionary Select Sector SPDR
  Rows: 6757
  Columns: 95
  Date range: 1998-12-22 00:00:00 to 2025-10-31 00:00:00

[Step 2/10] Computing Features...

Extracting pre-computed features for Consumer Discretionary Select Sector SPDR...
  Features shape: (6757, 95)

[Step 3/10] Loading KAMA+MSR Labels...

LOADING KAMA+MSR LABELS FOR Consumer Discretionary Select Sector SPDR
Model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20251007
Loading from: Consumer Discretionary Select Sector SPDR_KAMA-MSR_4-regimes.pkl
✓ Loaded labels for: Consumer Discretionary Select Sector SPDR
  Label date range: 1998-12-23 00:00:00 to 2025-10-07 00:00:00
  Total periods: 6738

  Original 4-regime distribution:
    0 -   LV Bullish:  4356 ( 64.9%)
    1 -   LV Bearish:  1502 ( 22.4%)
    2 -   HV Bullish:   112 (  1.7%)
    3 -   HV Bearish:   743 ( 11.1%)

[Step 4/10] Using Original 4-Regime Labels...

[Step 5-9/10]

[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:   53.8s


Loaded data for: iShares Russell 2000 ETF
  Rows: 6397
  Columns: 95
  Date range: 2000-05-26 00:00:00 to 2025-10-31 00:00:00

[Step 2/10] Computing Features...

Extracting pre-computed features for iShares Russell 2000 ETF...
  Features shape: (6397, 95)

[Step 3/10] Loading KAMA+MSR Labels...

LOADING KAMA+MSR LABELS FOR iShares Russell 2000 ETF
Model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20251007
Loading from: iShares Russell 2000 ETF_KAMA-MSR_4-regimes.pkl
✓ Loaded labels for: iShares Russell 2000 ETF
  Label date range: 2000-05-30 00:00:00 to 2025-10-07 00:00:00
  Total periods: 6378

  Original 4-regime distribution:
    0 -   LV Bullish:  3523 ( 55.3%)
    1 -   LV Bearish:  2380 ( 37.4%)
    2 -   HV Bullish:   109 (  1.7%)
    3 -   HV Bearish:   353 (  5.5%)

[Step 4/10] Using Original 4-Regime Labels...

[Step 5-9/10] Preparing Training Data...
  - Loading cross-asset features: False
  - Including mac

[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:  1.8min


✗ Error training Invesco DB Agriculture: "None of [MultiIndex([('Invesco DB Agriculture',  'open'),\n            ('Invesco DB Agriculture',  'high'),\n            ('Invesco DB Agriculture',   'low'),\n            ('Invesco DB Agriculture', 'close')],\n           )] are in the [columns]"
Starting training for Health Care Select Sector SPDR...
KMRF model initialized
  Asset: Health Care Select Sector SPDR
  Asset class: us_equity
  Classification type: original
  Training end date: 20251007
  Test end date: All available data
  Data type: master
  Data path: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
  KAMA+MSR model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20251007
  Validation/Test split: Will be calculated after loading data
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
  Feature selection: Boruta=False, Consensus=False
  Custom XGB para

AttributeError: 'Parallel' object has no attribute '_pre_dispatch_amount'